In [ ]:
import pandas as pd
import plotly.express as px
from rouge_metric import PyRouge
rouge = PyRouge( rouge_l=True)

In [89]:
## Read data
data_eval=pd.read_json('../../data/evaluation/eval_generator_dataset_full.jsonl',lines=True)


In [23]:
rouge = PyRouge( rouge_l=True)

In [90]:
rouge_data = []
for i,row in data_eval.iterrows():
    print(f'processing row {i}')
    score = rouge.evaluate(
        [str(row['generated_answer'])],[[str(row['ground_truth_answer'])]]
    )
    rouge_data.append(score['rouge-l'])
    

processing row 0
processing row 1
processing row 2
processing row 3
processing row 4
processing row 5
processing row 6
processing row 7
processing row 8
processing row 9
processing row 10
processing row 11
processing row 12
processing row 13
processing row 14
processing row 15
processing row 16
processing row 17
processing row 18
processing row 19
processing row 20
processing row 21
processing row 22
processing row 23
processing row 24
processing row 25
processing row 26
processing row 27
processing row 28
processing row 29
processing row 30
processing row 31
processing row 32
processing row 33
processing row 34
processing row 35
processing row 36
processing row 37
processing row 38
processing row 39
processing row 40
processing row 41
processing row 42
processing row 43
processing row 44
processing row 45
processing row 46
processing row 47


In [91]:
rouge_evaluation = pd.DataFrame(
    rouge_data
)

In [92]:
rouge_evaluation

,r,p,f
0,0.409091,0.126761,0.193548
1,0.875000,0.333333,0.482759
2,0.078431,0.090909,0.084211
3,0.206897,0.240000,0.222222
4,0.565217,0.110169,0.184397
5,0.230769,0.166667,0.193548
6,0.416667,0.099338,0.160428
7,0.357143,0.172414,0.232558
8,0.571429,0.131868,0.214286
9,0.562500,0.341772,0.425197


In [93]:
import plotly.graph_objects as go

fig = go.Figure()

# =========================
# Main ROUGE curves
# =========================
fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['r'] * 100,
        mode="lines+markers",
        name="ROUGE-L Recall",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['p'] * 100,
        mode="lines+markers",
        name="ROUGE-L Precision",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['f'] * 100,
        mode="lines+markers",
        name="ROUGE-L F1",
        line=dict(width=4),
        marker=dict(size=8),
    )
)

# =========================
# Mean lines
# =========================
mean_f1 = rouge_evaluation['f'].mean() * 100
mean_p = rouge_evaluation['p'].mean() * 100
mean_r = rouge_evaluation['r'].mean() * 100

fig.add_hline(
    y=mean_f1,
    line_width=3,
    line_dash="dash",
    annotation_text=f"Mean F1 = {mean_f1:.2f}%",
    annotation_position="top left"
)

fig.add_hline(
    y=mean_p,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Precision = {mean_p:.2f}%",
    annotation_position="bottom left"
)

fig.add_hline(
    y=mean_r,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Recall = {mean_r:.2f}%",
    annotation_position="bottom right"
)


# =========================
# Layout
# =========================
fig.update_layout(
    title={
        "text": "ROUGE-L Evaluation of the Quran RAG Generator",
        "x": 0.5,
        "xanchor": "center",
        "font": dict(size=24)
    },
    xaxis_title="Evaluation Samples",
    yaxis_title="Score (%)",
    template="plotly_white",
    
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    font=dict(size=15)
)

# Better axes
fig.update_xaxes(
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    range=[0, 100],
    showgrid=True,
    gridwidth=1
)

fig.show()

In [109]:
def print_case(title, df, data_eval):
    print("\n" + "="*80)
    print(title)
    print("="*80)

    for i, (idx, row) in enumerate(df.iterrows()):
        original = data_eval.loc[idx]

        print(f"\n📌 Case {i+1} | Index: {idx}")
        print("-"*80)

        print("\n🔹 Ground Truth Answer:")
        print(original['ground_truth_answer'])

        print("\n🔸 Generated Answer:")
        print(original['generated_answer'])

        print("\n📊 Scores:")
        print(f"Precision: {row['p']:.3f}")
        print(f"Recall:    {row['r']:.3f}")
        print(f"F1:        {row['f']:.3f}")

In [110]:
# =========================
# LOW PRECISION (Top 3 worst)
# =========================
low_precision = rouge_evaluation.nsmallest(3, 'p')

# =========================
# LOW RECALL (Top 3 worst)
# =========================
low_recall = rouge_evaluation.nsmallest(3, 'r')

# =========================
# HIGH PRECISION (Top 3 best)
# =========================
high_precision = rouge_evaluation.nlargest(3, 'p')

# =========================
# HIGH RECALL (Top 3 best)
# =========================
high_recall = rouge_evaluation.nlargest(3, 'r')

In [111]:
print_case("🔴 LOW PRECISION CASES (Hallucination / verbosity issues)", low_precision, data_eval)

print_case("🔵 LOW RECALL CASES (Retrieval failure)", low_recall, data_eval)

print_case("🟢 HIGH PRECISION CASES (Good conciseness)", high_precision, data_eval)

print_case("🟣 HIGH RECALL CASES (Good coverage)", high_recall, data_eval)


🔴 LOW PRECISION CASES (Hallucination / verbosity issues)

📌 Case 1 | Index: 22
--------------------------------------------------------------------------------

🔹 Ground Truth Answer:
The three steps are: (1) admonish/advise her, (2) abandon her in bed, and (3) beat her (lightly, without severe beating).

🔸 Generated Answer:
Direct Answer: The Quran does not explicitly mention specific steps for a husband to take when facing his wife's ill-conduct (nushuz). However, Surah 4:34 provides guidance on the husband's right to chastise his disobedient wife.

Quran References: Surah 4:34

Tafsir Insights: In Surah 4:34, it is mentioned that a disobedient wife may be admonished, then left in bed (separated) before being beaten, but only lightly. It is important to note that the verses emphasize the necessity of fairness and justice in such situations.

1. Admonish her: The husband should initially admonish his disobedient wife and encourage her to return to righteous behavior.
2. Separate from